
# Blog Factory — Ideas Browser (DuckDB)

Bu notebook, DuckDB içindeki **ideas**, **idea_products** ve **v_products** tablolarını rahatça gezmenizi sağlar.

## Neler yapabilirsiniz?
- Seçtiğiniz kategoriye göre **son eklenen fikirleri** tam `idea_id` ve başlıklarıyla listeleyin.
- Bir `idea_id` girip **bağlı ürünleri** (merkez + rakipler) başlıklarıyla görün.
- Seçtiğiniz `ASIN` için **hangi fikirler** var, hızlıca bulun.

> **Not:** `DB_PATH` değişkenini gerektiğinde güncelleyin.


In [ ]:

# === Setup ===
DB_PATH = "warehouse/blog_factory.duckdb"  # gerekirse mutlak path verin: "/home/ubuntu/blog-factory/warehouse/blog_factory.duckdb"

import duckdb
import pandas as pd

con = duckdb.connect(DB_PATH)
print("Connected ✓", DB_PATH)


In [ ]:

def list_recent_ideas(category: str = None, limit: int = 20) -> pd.DataFrame:
    """Kategoriye göre (opsiyonel) son eklenen fikirleri tam idea_id ile getirir."""
    if category:
        q = '''
        SELECT idea_id,
               idea_title,
               category_slug,
               created_at
        FROM ideas
        WHERE category_slug = ?
        ORDER BY created_at DESC
        LIMIT ?
        '''
        return con.execute(q, [category, limit]).df()
    else:
        q = '''
        SELECT idea_id,
               idea_title,
               category_slug,
               created_at
        FROM ideas
        ORDER BY created_at DESC
        LIMIT ?
        '''
        return con.execute(q, [limit]).df()


def show_idea_products(idea_id: str) -> pd.DataFrame:
    """Bir fikre bağlı tüm ürünleri (ASIN + başlık + marka + kategori) döndürür."""
    q = '''
    SELECT ip.parent_asin,
           vp.product_title,
           COALESCE(vp.brand, '') AS brand,
           COALESCE(vp.category_slug, '') AS category_slug
    FROM idea_products ip
    JOIN v_products vp ON vp.parent_asin = ip.parent_asin
    WHERE ip.idea_id = ?
    ORDER BY vp.brand, vp.product_title
    '''
    return con.execute(q, [idea_id]).df()


def ideas_for_asin(parent_asin: str, limit: int = 50) -> pd.DataFrame:
    """Belirli bir ASIN için hangi fikirler var, başlıklarıyla listeler."""
    q = '''
    SELECT i.idea_id, i.idea_title, i.category_slug, i.created_at
    FROM ideas i
    JOIN idea_products ip ON ip.idea_id = i.idea_id
    WHERE ip.parent_asin = ?
    ORDER BY i.created_at DESC
    LIMIT ?
    '''
    return con.execute(q, [parent_asin, limit]).df()



## Örnek 1 — Son 10 **electronics** fikri (tam `idea_id` ile)


In [ ]:

df_ideas = list_recent_ideas(category="electronics", limit=10)
df_ideas



## Örnek 2 — Bir `idea_id` seçip ürünlerini listele
Aşağıdaki örnekte `idea_id` değerini kendi listenizden kopyalayıp girin.


In [ ]:

# örnek: idea_id = "i-the-journey-of-sound-how-quality-wiring-kits-enhance-your-car-audio"
idea_id = "PASTE_ID_HERE"
if idea_id != "PASTE_ID_HERE":
    show_idea_products(idea_id)
else:
    print("Bir idea_id girin ve hücreyi tekrar çalıştırın.")



## Örnek 3 — Bir ASIN için ilişkili fikirler


In [ ]:

# örnek: parent_asin = "B08BG7J2MN"
parent_asin = "PASTE_ASIN_HERE"
if parent_asin != "PASTE_ASIN_HERE":
    ideas_for_asin(parent_asin)
else:
    print("Bir parent_asin girin ve hücreyi tekrar çalıştırın.")
